# Effect of one threshold-link agent

This notebook compares a treatment with one threshold-link agent against a paired all-best-response baseline. Both agent types have unit activation weight, so the treatment changes policy assignment without changing selection frequency. Each pair shares the sampled parameters and simulator seed.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401
from matplotlib_inline.backend_inline import set_matplotlib_formats
from statsmodels.nonparametric.smoothers_lowess import lowess

ROOT = Path.cwd().resolve()
if not (ROOT / "tasks").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tasks.policies import load_policy_results

RUN = ROOT / "artifacts/simulation_runs/run-20260729T021448.zarr"
CONFIG = ROOT / "configs/simulation.best_response_threshold_link.single_threshold_agent.json"
FIGURE_DIR = ROOT / "artifacts/figures" / RUN.stem
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
WELFARE = "centered_mean_welfare.mean"
COMPOSITION = "threshold_link_count"
COLORS = {10: "#3d405b", 50: "#e07a5f", 100: "#2a9d8f"}

plt.style.use("science")
set_matplotlib_formats("svg")
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "legend.title_fontsize": 13,
    "figure.titlesize": 19,
})
pd.set_option("display.max_columns", 10)

In [ ]:
run_config = json.loads((RUN / "run_config.json").read_text())
checked_config = json.loads(CONFIG.read_text())
contract_sections = ("sampling", "simulator", "game", "population", "observers")
if any(run_config[name] != checked_config[name] for name in contract_sections):
    raise ValueError("The selected artifact was not produced by the current config")

activation = run_config["population"]["agent_parameters"]["activation_weight"]
policy_overrides = [
    policy for policy in run_config["population"]["policies"]
    if "activation_weight" in policy.get("agent_parameters", {})
]
if activation != {"kind": "fixed", "value": 1.0} or policy_overrides:
    raise ValueError("This analysis requires equal unit activation weights")

ds = load_policy_results(RUN, stats=[WELFARE], normalize_time=True)
if ds.attrs["has_failures"]:
    raise ValueError("The source run contains failed rows")
if not np.array_equal(ds.coords[COMPOSITION], [0, 1]):
    raise ValueError("Expected paired threshold-link counts [0, 1]")

baseline = ds["x"].sel(observable=WELFARE, **{COMPOSITION: 0})
treatment = ds["x"].sel(observable=WELFARE, **{COMPOSITION: 1})
delta = treatment - baseline
np.testing.assert_allclose(
    ds["theta"].sel(**{COMPOSITION: 0}),
    ds["theta"].sel(**{COMPOSITION: 1}),
)

print(
    f"{ds.sizes['parameter_row']:,} paired parameter rows per network size; "
    f"{run_config['simulator']['n_episodes']} episodes per row; "
    f"common horizon = {float(ds.step[-1]):g} activations per agent."
)

## Endpoint effect

The paired effect is treatment welfare minus baseline welfare. Quantiles and proportions summarize heterogeneity over the fixed parameter design; they are not confidence intervals.

In [ ]:
summary_rows = []
for size in ds.n_agents.values:
    baseline_endpoint = baseline.sel(n_agents=size).isel(step=-1).values
    treatment_endpoint = treatment.sel(n_agents=size).isel(step=-1).values
    effect = delta.sel(n_agents=size).isel(step=-1).values
    q05, median, q95 = np.quantile(effect, [0.05, 0.50, 0.95])
    summary_rows.append({
        "n_agents": int(size),
        "Baseline mean": baseline_endpoint.mean(),
        "Treatment mean": treatment_endpoint.mean(),
        "Mean effect": effect.mean(),
        "Median effect": median,
        "5% effect": q05,
        "95% effect": q95,
        "P(effect > 0)": np.mean(effect > 0),
    })

endpoint_summary = pd.DataFrame(summary_rows).set_index("n_agents")
formats = {column: "{:.4f}" for column in endpoint_summary.columns}
formats["P(effect > 0)"] = "{:.1%}"
endpoint_summary.style.format(formats)

In [ ]:
fig, axes = plt.subplots(
    1, ds.sizes["n_agents"], figsize=(13.5, 4.3),
    sharex=True, sharey=True, constrained_layout=True,
)
for size, ax in zip(ds.n_agents.values, axes, strict=True):
    values = delta.sel(n_agents=size).values
    q05, q95 = np.quantile(values, [0.05, 0.95], axis=0)
    color = COLORS[int(size)]
    ax.fill_between(
        ds.step, q05, q95, color=color, alpha=0.16, linewidth=0,
        label=r"5\%--95\% range", zorder=1,
    )
    ax.plot(
        ds.step, values.mean(axis=0), color=color, lw=2.5,
        label="mean", zorder=3,
    )
    ax.axhline(0, color="black", lw=0.8, zorder=2)
    ax.set_title(rf"$n={int(size)}$")
    ax.set_xlim(float(ds.step[0]), float(ds.step[-1]))
    ax.set_axisbelow(True)
    ax.grid(which="major", color="#b7b7b7", alpha=0.28, linewidth=0.6)
fig.supxlabel("activation opportunities per agent")
fig.supylabel("paired welfare effect")
axes[0].legend(loc="upper right", frameon=False)
fig.suptitle("Paired welfare effect over time")
dynamics_path = FIGURE_DIR / "threshold_agent_effect_dynamics.pdf"
fig.savefig(dynamics_path, format="pdf", bbox_inches="tight")
print(f"Saved: {dynamics_path.relative_to(ROOT)}")
plt.show()

## Parameter heterogeneity

For continuous inputs, lines are LOWESS conditional means and bands are local weighted 5–95% ranges. Visibility uses mean points with 5–95% bars at its integer levels. The ranges describe heterogeneity over the remaining parameter space; they are not confidence intervals.

In [ ]:
def weighted_quantiles(values, weights, probabilities=(0.05, 0.95)):
    order = np.argsort(values)
    values = np.asarray(values)[order]
    weights = np.asarray(weights)[order]
    positions = (np.cumsum(weights) - 0.5 * weights) / weights.sum()
    return np.interp(probabilities, positions, values)


def continuous_effect_profile(x, y, *, frac=0.2, n_points=151):
    order = np.argsort(x)
    x = np.asarray(x, dtype=float)[order]
    y = np.asarray(y, dtype=float)[order]
    grid = np.linspace(x[0], x[-1], n_points)
    fitted = lowess(
        y, x, frac=frac, it=0, delta=(x[-1] - x[0]) / n_points,
        return_sorted=True,
    )
    mean = np.interp(grid, fitted[:, 0], fitted[:, 1])

    window = min(len(x), max(3, int(np.ceil(frac * len(x)))))
    lower = np.empty_like(grid)
    upper = np.empty_like(grid)
    for index, center in enumerate(grid):
        insertion = np.searchsorted(x, center)
        start = int(np.clip(insertion - window // 2, 0, len(x) - window))
        local_x = x[start : start + window]
        local_y = y[start : start + window]
        distance = np.abs(local_x - center)
        scale = distance.max()
        weights = (1 - (distance / scale) ** 3) ** 3 if scale else np.ones(window)
        lower[index], upper[index] = weighted_quantiles(local_y, weights)
    return grid, mean, lower, upper


theta = (
    ds["theta"]
    .sel(n_agents=ds.n_agents.values[0], **{COMPOSITION: 0})
    .to_pandas()
)
features = {
    "visibility_distance": "Visibility distance",
    "threshold_link.link_threshold": "Link threshold",
    "best_response.outside_optimism": "Outside optimism",
    "best_response.exploration": "Exploration",
}
sizes = ds.n_agents.values
offsets = np.linspace(-0.12, 0.12, len(sizes))

fig, axes = plt.subplots(
    2, 2, figsize=(12.5, 7.8), sharey=True, constrained_layout=True
)
for (feature, label), ax in zip(features.items(), axes.flat, strict=True):
    x = theta[feature].to_numpy()
    for size_index, size in enumerate(sizes):
        effect = delta.sel(n_agents=size).isel(step=-1).values
        color = COLORS[int(size)]
        if feature == "visibility_distance":
            levels = np.sort(np.unique(x))
            grouped = [effect[x == level] for level in levels]
            means = np.asarray([values.mean() for values in grouped])
            ranges = np.asarray(
                [np.quantile(values, [0.05, 0.95]) for values in grouped]
            )
            ax.errorbar(
                levels + offsets[size_index], means,
                yerr=np.vstack((means - ranges[:, 0], ranges[:, 1] - means)),
                fmt="o", ms=5.5, capsize=3, elinewidth=1.2,
                markeredgecolor="white", markeredgewidth=0.7, color=color,
                label=rf"$n={int(size)}$", zorder=3,
            )
            ax.set_xticks(levels)
            ax.set_xlim(levels[0] - 0.45, levels[-1] + 0.45)
        else:
            grid, mean, lower, upper = continuous_effect_profile(x, effect)
            ax.fill_between(
                grid, lower, upper, color=color, alpha=0.12,
                linewidth=0, zorder=1,
            )
            ax.plot(
                grid, mean, color=color, lw=2.3,
                label=rf"$n={int(size)}$", zorder=3,
            )
            ax.set_xlim(grid[0], grid[-1])
            tick_max = 0.1 if feature.endswith("exploration") else 1.0
            ax.set_xticks(np.linspace(0.0, tick_max, 6))
    ax.axhline(0, color="black", lw=0.8, zorder=2)
    ax.set_xlabel(label)
    ax.set_axisbelow(True)
    ax.grid(which="major", color="#b7b7b7", alpha=0.28, linewidth=0.6)

fig.supylabel("paired endpoint welfare effect")
handles, labels = axes[0, 1].get_legend_handles_labels()
fig.legend(
    handles, labels, title="Network size",
    loc="outside right center", frameon=False,
)
fig.suptitle("Endpoint welfare effect across sampled parameters")
profiles_path = FIGURE_DIR / "threshold_agent_effect_parameter_profiles.pdf"
fig.savefig(profiles_path, format="pdf", bbox_inches="tight")
print(f"Saved: {profiles_path.relative_to(ROOT)}")
plt.show()

## Reporting notes

- The design is a scrambled Sobol sample, not a random sample or a Saltelli sensitivity design.
- Outputs are means over 16 episodes, so episode-level simulation variance is unavailable.
- Time is population-average activations per agent. Unit activation weights make expected exposure equal across agents.
- The analysis describes this artifact's paired parameter rows; it does not report sampling uncertainty.